# Lồng tiếng clip MFCC bằng VieNeu-TTS v3 Turbo

Chạy trên runtime CPU là đủ (ONNX Runtime). Đầu vào: `/content/tts/<clip>.json` (file trong `narration/`), mỗi đoạn có `id`, `sub` (phụ đề) và `say` (câu đọc, đã viết thuật ngữ theo cách đọc).
Đầu ra: `/content/tts/out/<clip>/<id>.wav` (48 kHz, mono, PCM 16 bit, đã cắt khoảng lặng hai đầu) và `durations.json`.

In [ ]:
# Newer onnxruntime rejects the model's external weight file inside the HF cache (blob symlinks), 1.22 loads it.
# Restart the session after this cell if onnxruntime was already imported.
!pip install -q vieneu onnxruntime==1.22.1

In [ ]:
CLIP = '01-van-de'
VOICE = 'Ngọc Huyền'
GAIN_DBFS = -20.0

In [ ]:
import json, os, time
import numpy as np, soundfile as sf
from vieneu import Vieneu

tts = Vieneu()
SR = 48000

def tidy(x, sr=SR):
    x = np.asarray(x, dtype=np.float32).squeeze()
    hop = sr // 100
    rms = np.array([np.sqrt(np.mean(x[i:i + hop] ** 2) + 1e-12) for i in range(0, len(x), hop)])
    on = np.where(20 * np.log10(rms) > 20 * np.log10(rms.max()) - 40)[0]
    if len(on):
        x = x[max(0, (on[0] - 3) * hop): min(len(x), (on[-1] + 4) * hop)]
    x = x * 10 ** (GAIN_DBFS / 20) / (np.sqrt(np.mean(x ** 2)) + 1e-9)
    return np.clip(x, -0.98, 0.98)

def synth(text, voice=VOICE):
    return tidy(tts.infer(text, voice=voice))

In [ ]:
spec = json.load(open(f'/content/tts/{CLIP}.json'))
out = f'/content/tts/out/{CLIP}'
os.makedirs(out, exist_ok=True)
dur = {}
for seg in spec['segments']:
    t = time.time()
    x = synth(seg['say'])
    sf.write(f"{out}/{seg['id']}.wav", x, SR, subtype='PCM_16')
    dur[seg['id']] = round(len(x) / SR, 3)
    print(f"{seg['id']:>8}  {dur[seg['id']]:6.2f} s  ({time.time() - t:.1f} s)", flush=True)
json.dump({'voice': VOICE, 'sr': SR, 'durations': dur}, open(f'{out}/durations.json', 'w'), ensure_ascii=False, indent=1)
!cd /content/tts/out && tar -czf /content/tts/{CLIP}.tgz {CLIP}
print('total', round(sum(dur.values()), 1), 's')

## Nghe thử các giọng (tùy chọn)
Cùng một câu cho vài giọng, để chọn `VOICE` cho các clip sau.

In [ ]:
SAMPLE = 'Bộ lọc Mel chỉ gộp các bin lại, nó không tách cái lược ra khỏi cái khuôn.'
os.makedirs('/content/tts/voices', exist_ok=True)
for v in ['Ngọc Huyền', 'Trúc Ly', 'Mai Anh', 'Thùy Dung', 'Hải Đăng', 'Thiện Minh']:
    sf.write(f'/content/tts/voices/{v}.wav', synth(SAMPLE, v), SR, subtype='PCM_16')
!cd /content/tts && tar -czf voices.tgz voices